# SSL: inference for five models

Use the existing **soil-grain-size-inference** Kaggle notebook.

**Inputs:** competition `soil-grain-size-from-photos` and the completed notebook
`denysmykhailov/soil-ssl-part-3-dinov3-mae`, **version 8** (its saved Outputs).
The Outputs already contain weights and the matching `soilcv` source;
`soil-cv-utils`, official pretrained weights and W&B secrets are not needed.

Select **2 × T4** and Internet. This notebook only predicts and saves CSVs.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

os.environ["USE_TF"] = "0"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1"])

import pandas as pd
import torch

studies = [p.parent.parent for p in Path("/kaggle/input").rglob("scores.csv")
           if (p.parent.parent / "source/soilcv/ssl/inference.py").is_file()]
assert len(studies) == 1, "Attach the completed SSL notebook Outputs, version 8."
STUDY = studies[0]
sys.path.insert(0, str(STUDY / "source"))

from soilcv.config import seed_everything
from soilcv.curves import make_submission
from soilcv.data import load_metadata
from soilcv.predictions import mean_curves
from soilcv.ssl.analysis import expected_experiments
from soilcv.ssl.inference import predict_fold

assert torch.cuda.device_count() == 2, "Select 2 × T4."
torch.set_num_threads(2)
seed_everything(42)
OUT = Path("/kaggle/working/ssl_inference")
OUT.mkdir(parents=True, exist_ok=True)

## Models

Four best remaining CV recipes plus the requested `dinov3-adapted-lp`.
Each prediction averages four fold checkpoints.

In [ ]:
MODELS = [
    "mae-lpft",
    "mae-full",
    "dinov3-adapted-full",
    "dinov3-lpft-no-color",
    "dinov3-adapted-lp",
]
scores = pd.read_csv(STUDY / "results/scores.csv").set_index("experiment").loc[MODELS]
display(scores[["cv_emd", "cv_std"]].round(3))

DATA = next(Path("/kaggle/input").rglob("Training_labels_updated.csv")).parent
_, photos, template = load_metadata(DATA)
photos = photos[photos.split.eq("test")].copy()
assert set(photos.sample_id) == set(template.sample_id)
recipes = expected_experiments()

## Both GPUs

GPU 0 predicts folds 0 and 2; GPU 1 predicts folds 1 and 3.
Each GPU has its own tile cache. Exceptions stop submission export.

In [ ]:
def infer_folds(gpu):
    device = f"cuda:{gpu}"
    cache = Path(f"/tmp/ssl-inference-{gpu}")
    with torch.cuda.device(gpu):
        for name in MODELS:
            encoder, source, strategy = recipes[name]
            for fold in range(gpu, 4, 2):
                folder = STUDY / "folds" / encoder / f"fold_{fold}" / source / strategy
                prediction = predict_fold(folder, photos, cache, device=device, amp=True)
                prediction.to_csv(OUT / f"{name}_fold{fold}.csv", index=False)
                print(f"GPU {gpu}: {name}, fold {fold} complete", flush=True)

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(infer_folds, gpu) for gpu in range(2)]
    for future in futures:
        future.result()

## Five separate submissions

One CSV per model is saved directly in `/kaggle/working/`.
Each file averages only that model's four folds; models are not mixed.
`make_submission` aligns test IDs and validates the cumulative curves.
Nothing is submitted to the leaderboard automatically.


In [ ]:
for name in MODELS:
    folds = [pd.read_csv(OUT / f"{name}_fold{fold}.csv") for fold in range(4)]
    path = OUT.parent / f"submission_{name}.csv"
    submission = make_submission(mean_curves(folds), template, path=path)
    print(f"{path.name}: {len(submission)} test samples → {path}")
